# QC_Run_Validation

Runs the Quality Catalog and writes `dq_run_results`, `dq_violations` and `dq_execution_metrics`. Schedule this nightly, after the source tables have refreshed.


In [ ]:
%run QC_Config


In [ ]:
%run QC_Rules


In [ ]:
%run QC_Engine


In [ ]:
# =============================================================================
# Cell source for the QC_Run_Validation notebook.
# Runs the Quality Catalog validation engine and prints the run evidence.
#
# Schedule this notebook nightly, after the source tables have refreshed.
# =============================================================================

from __future__ import annotations

from pyspark.sql import SparkSession


def _table_exists(spark, table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).count()
        return True
    except Exception:
        return False


def print_run_evidence(config_mapping: dict) -> None:
    """Show the latest execution metrics, rule-group summary, and open violations."""
    spark = SparkSession.builder.getOrCreate()

    config = build_settings(
        config_mapping,
        [
            "DEFAULT_SCHEMA",
            "DQ_RESULTS_TABLE",
            "DQ_VIOLATIONS_TABLE",
            "DQ_EXECUTION_METRICS_TABLE",
        ],
        "QUALITY_CATALOG_CONFIG",
    )
    schema = config.DEFAULT_SCHEMA

    results_table = f"{schema}.{config.DQ_RESULTS_TABLE}"
    violations_table = f"{schema}.{config.DQ_VIOLATIONS_TABLE}"

    # write_execution_metric falls back to an unqualified table name when the
    # namespace is not resolvable, so look for the metrics table in both places.
    metrics_table = next(
        (
            candidate
            for candidate in (
                f"{schema}.{config.DQ_EXECUTION_METRICS_TABLE}",
                config.DQ_EXECUTION_METRICS_TABLE,
            )
            if _table_exists(spark, candidate)
        ),
        None,
    )

    print("Detected tables:")
    print("  results_table:", results_table if _table_exists(spark, results_table) else None)
    print("  violations_table:", violations_table if _table_exists(spark, violations_table) else None)
    print("  metrics_table:", metrics_table)

    if metrics_table:
        print("\nLatest execution metrics rows:")
        spark.sql(
            f"""
            SELECT
                script_name,
                status,
                row_count,
                started_at_utc,
                finished_at_utc,
                duration_seconds,
                is_retryable,
                error_message
            FROM {metrics_table}
            ORDER BY finished_at_utc DESC
            LIMIT 10
            """
        ).show(truncate=False)

    if _table_exists(spark, results_table):
        latest = spark.sql(
            f"SELECT run_id FROM {results_table} ORDER BY run_timestamp DESC LIMIT 1"
        ).collect()
        if latest:
            run_id = latest[0]["run_id"]
            print(f"\nRule-group summary for latest run_id: {run_id}")
            # Pass run_id via a temp view rather than string interpolation to
            # prevent second-order SQL injection from crafted values in the table.
            spark.createDataFrame([(run_id,)], ["_run_id"]).createOrReplaceTempView("_ev_run_id")
            spark.sql(
                f"""
                SELECT
                    rule_group,
                    COUNT(*) AS total_rules,
                    SUM(CASE WHEN status = 'PASSED' THEN 1 ELSE 0 END) AS passed,
                    SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) AS failed,
                    SUM(CASE WHEN status = 'ERROR' THEN 1 ELSE 0 END) AS errors
                FROM {results_table}
                WHERE run_id = (SELECT _run_id FROM _ev_run_id)
                GROUP BY rule_group
                ORDER BY rule_group
                """
            ).show(truncate=False)
            try:
                spark.catalog.dropTempView("_ev_run_id")
            except Exception:
                pass  # best-effort cleanup; non-fatal

    if _table_exists(spark, violations_table):
        print("\nCurrent violations by issue_status:")
        spark.sql(
            f"""
            SELECT issue_status, COUNT(*) AS cnt
            FROM {violations_table}
            GROUP BY issue_status
            ORDER BY cnt DESC
            """
        ).show(truncate=False)


In [ ]:
configure(QUALITY_CATALOG_CONFIG, QUALITY_CATALOG_RUNTIME)

results_count, violations_count = run_with_metrics(RULE_CATALOG_SOURCES, "run_validation")
print(f"\nResult rows: {results_count}   Violations processed: {violations_count}")

print_run_evidence(QUALITY_CATALOG_CONFIG)
